In [1]:
%matplotlib widget
%load_ext autoreload
%autoreload 2
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import HTML

In [2]:
import numpy as np
import torch
import hydra
from omegaconf import DictConfig, OmegaConf
import os
import random

np.random.seed(0)
torch.manual_seed(0)
random.seed(0)

In [3]:
def get_xela_pretraining_datasets(data_cfg: DictConfig):
    def get_xela_dataset(dataset_cfg: DictConfig, dataset_name: str, d_id: int, object_class: int):
        data_path = f"{dataset_cfg.data_path}"
        data_files = os.listdir(data_path)
        dataset_name_exists = True in [f in f"{dataset_name}" for f in data_files]
        if not dataset_name_exists:
            print(f"Dataset {dataset_name} not found")
            return None

        dataset = hydra.utils.instantiate(
            dataset_cfg,
            data_path=f"{data_path}/{dataset_name}/{d_id}",
            object_class=object_class,
        )
        return dataset
    
    val_datasets = []
    dataset_list = data_cfg.dataset_list
    object_classes = []
    object_class_sizes = []
    for dataset_l in dataset_list:
        if dataset_l.type == "teleop":
            val_dataset_ids = dataset_l.val_dataset_ids
            
            for obj in dataset_l.sequence_list:
                print(f"obj: {obj}")
                object_classes.append(obj)
                object_class_sizes.append(0)

                for d_id in val_dataset_ids:
                    val_datasets.append(
                        get_xela_dataset(
                            dataset_l.dataset, dataset_name=obj, d_id=d_id, object_class=len(object_classes) - 1
                        )
                    )
                    object_class_sizes[-1] += len(val_datasets[-1])
    return val_datasets

In [4]:
import torch.utils.data as data

with hydra.initialize(version_base="1.3", config_path="../config"):
    config = hydra.compose(
        config_name="default.yaml",
        overrides=[
            "+experiment=xela/dinov2",
            "paths=default",
            "hydra.job.num=1",
            "data.dataset_list.0.dataset.data_path=/home/akashsharma/workspace/datasets/xela/pretraining/extracted",
            "paths.output_dir='outputs/.'",
            "paths.work_dir='outputs/.'",
        ],
        return_hydra_config=True,
    )


val_datasets = get_xela_pretraining_datasets(config.data)
val_dset = data.ConcatDataset(val_datasets)
encoder = config.algorithm.encoder

obj: corn
obj: rubikscube
obj: metalcup
obj: cup
obj: ball
obj: drill
obj: mustard
obj: lego
obj: pringle
obj: legopool
obj: watermelon
obj: popcorn
obj: loofah
obj: slipper


In [5]:
def load_encoder(encoder_model, checkpoint_encoder: str, encoder_type: str):
   checkpoint = torch.load(checkpoint_encoder)
   if "jepa" in encoder_type:
       encoder_key = "target_encoder"
   elif "dino" in encoder_type:
       encoder_key = "teacher_encoder.backbone"
   else:
       encoder_key = "encoder"
   # get the keys in the checkpoint that contain the encoder
   target_keys = [key for key in checkpoint["model"].keys() if encoder_key in key]
   # remove the prefix from the keys
   new_keys = [key.replace(f"{encoder_key}.", "") for key in target_keys]
   # create a state_dict  with keys target_keys from the checkpoint
   new_state_dict = {
       new_key: checkpoint["model"][target_key] for new_key, target_key in zip(new_keys, target_keys)
   }
   # load the state_dict into the model
   encoder_model.load_state_dict(new_state_dict)
   print(f"Loaded encoder from {checkpoint_encoder}")
   return encoder_model

print(f"encoder: {encoder}")
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
ssl_encoder = hydra.utils.instantiate(encoder, normalization=None)   
ssl_encoder = ssl_encoder.to(device)
print(ssl_encoder) 

checkpoint_path = "/home/akashsharma/workspace/checkpoints/percepskin_models/dinov2_nopatch_sensorpose_500epochs.ckpt"
ssl_encoder = load_encoder(ssl_encoder, checkpoint_path, "dinov2")
ssl_encoder.eval()

encoder: {'_target_': 'tactile_ssl.model.xela_${model_size}', 'in_dim': 368, 'in_chans': 6, 'sequence_length': 10, 'time_chunk_size': 10, 'num_register_tokens': 1, 'pos_embed_fn': 'learned', 'with_masktoken': True, 'drop_path_rate': 0.3, 'drop_path_uniform': True, 'normalization': '${data.normalization}', 'causal': False}


/home/akashsharma/workspace/projects/tactile_ssl/tactile_ssl/model/layers/swiglu_ffn.py:43: UserWarning: xFormers is available (SwiGLU)
  warnings.warn("xFormers is available (SwiGLU)")
/home/akashsharma/workspace/projects/tactile_ssl/tactile_ssl/model/layers/attention.py:35: UserWarning: xFormers is available (Attention)
  warnings.warn("xFormers is available (Attention)")
/home/akashsharma/workspace/projects/tactile_ssl/tactile_ssl/model/layers/block.py:33: UserWarning: xFormers is available (Block)
  warnings.warn("xFormers is available (Block)")
/home/akashsharma/workspace/projects/tactile_ssl/tactile_ssl/model/layers/decoder_block.py:32: UserWarning: xFormers is available (Block)
  warnings.warn("xFormers is available (Block)")


Xela mean: tensor([0, 0, 0]), Xela std: tensor([1, 1, 1])
XelaTransformer(
  (blocks): ModuleList(
    (0-7): 8 x NestedTensorBlock(
      (norm1): LayerNorm((192,), eps=1e-06, elementwise_affine=True)
      (attn): MemEffAttention(
        (qkv): Linear(in_features=192, out_features=576, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=192, out_features=192, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): DropPath()
      (norm2): LayerNorm((192,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=192, out_features=768, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=768, out_features=192, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
      (ls2): Identity()
      (drop_path2): DropPath()
    )
  )
  (norm): LayerNorm((192,), eps=1e-06, elementwise_affine=True)
  (head): Identity(

XelaTransformer(
  (blocks): ModuleList(
    (0-7): 8 x NestedTensorBlock(
      (norm1): LayerNorm((192,), eps=1e-06, elementwise_affine=True)
      (attn): MemEffAttention(
        (qkv): Linear(in_features=192, out_features=576, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=192, out_features=192, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): DropPath()
      (norm2): LayerNorm((192,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=192, out_features=768, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=768, out_features=192, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
      (ls2): Identity()
      (drop_path2): DropPath()
    )
  )
  (norm): LayerNorm((192,), eps=1e-06, elementwise_affine=True)
  (head): Identity()
  (patch_embed): PatchEmbed1d(
    (proj): Conv1d(6, 192

In [6]:
from torch.utils.data import DataLoader
from tqdm import tqdm
import einops
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from tactile_ssl.data.xela.utils import xela_sensor_layout

val_dloader = DataLoader(val_dset, batch_size=256, shuffle=True, num_workers=0)

In [7]:
batches = []
for i in range(8): 
    batch = next(iter(val_dloader))
    batches.append(batch)

In [8]:
import umap.umap_ as umap
import numpy as np
import matplotlib.pyplot as plt

cls_tokens = []
patch_tokens = []
object_classes = []
sensor_data = []
with torch.no_grad():
    for batch in batches:
        x = batch['sensor'].to(device)
        object_class = batch['object_classification']
        features = ssl_encoder.forward_features(x)
        patch_features = features['x_norm_patchtokens']
        cls_token = features['x_norm_regtokens']

        cls_tokens.append(cls_token.cpu().numpy())
        patch_tokens.append(patch_features.cpu().numpy())
        object_classes.append(object_class)
        sensor_data.append(x.cpu().numpy())
        del features

cls_tokens = np.concatenate(cls_tokens, axis=0)
patch_tokens = np.concatenate(patch_tokens, axis=0)
object_classes = np.concatenate(object_classes, axis=0)
sensor_data = np.concatenate(sensor_data, axis=0)

print(f"cls_tokens: {cls_tokens.shape}")
print(f"patch_tokens: {patch_tokens.shape}")



cls_tokens: (2048, 1, 192)
patch_tokens: (2048, 368, 192)


In [9]:
reducer = umap.UMAP(random_state=42)
embedding = reducer.fit_transform(einops.rearrange(cls_tokens, 'b p c -> (b p) c'))

print(embedding.shape)
print(embedding)


string = ["corn",
       "rubikscube",
       "metalcup",
       "cup",
       "ball",
       "drill",
       "mustard",
       "lego",
       "pringle",
       "pool-of-lego",
       "watermelon",
       "popcorn",
       "loofah",
       "slipper"]
# string = ["corn", "rubikscube", "metalcup", "cup", "ball", "drill", "watermelon", "mustard", "lego", "pringle", "popcorn", "loofah", "pool-of-lego", "slipper"]

/home/akashsharma/miniforge3/envs/tactile_ssl/lib/python3.9/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


(2048, 2)
[[-7.8067775  7.1869273]
 [-2.8661063 -4.802152 ]
 [16.97621    7.4438405]
 ...
 [-6.807649   4.1289053]
 [-2.7262328 -3.9891865]
 [ 2.9998517 18.831745 ]]


In [ ]:

plt.rcParams.update({
    "text.usetex": True,
})


fig, ax = plt.subplots(figsize=(8, 6))
cax = ax.scatter(embedding[:, 0], embedding[:, 1], c=object_classes, cmap='tab20')

# sns.scatterplot(x="x", y="y",
#                 hue=object_classes,
#                 data=umap_data, ax=ax)
# plt.title("UMAP of CLS tokens")
# plt.show()
cbar = fig.colorbar(cax, boundaries=np.arange(len(string)+1)-0.5, ticks=np.arange(len(string)))
cbar.ax.set_yticklabels(string)
# plt.savefig("umap_cls_tokens.png")
plt.close(fig)
plt.show()

No such comm: f83db8af5f33465184b05e795f81a00d
No such comm: f83db8af5f33465184b05e795f81a00d
No such comm: f83db8af5f33465184b05e795f81a00d
No such comm: f83db8af5f33465184b05e795f81a00d
No such comm: f83db8af5f33465184b05e795f81a00d
No such comm: f83db8af5f33465184b05e795f81a00d


In [11]:
patch_embedding = reducer.fit_transform(einops.rearrange(patch_tokens, 'b p c -> b (p c)'))

print(patch_embedding.shape)

/home/akashsharma/miniforge3/envs/tactile_ssl/lib/python3.9/site-packages/sklearn/manifold/_spectral_embedding.py:285: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


(2048, 2)


In [12]:

fig, ax = plt.subplots(figsize=(8, 6))
cax = ax.scatter(patch_embedding[:, 0], patch_embedding[:, 1], c=object_classes, cmap='tab20')
cbar = fig.colorbar(cax, boundaries=np.arange(len(string)+1)-0.5, ticks=np.arange(len(string)))
cbar.ax.set_yticklabels(string)
plt.savefig("umap_patch_tokens.svg")
plt.close(fig)
# plt.show()

In [25]:
from pathlib import Path
from tactile_ssl.data.xela_pose_estimation import RelativePoseDataset
def create_from_files(data_path, urdf_path, baseline_signal_path, config):
        dataset_ = []
        for stage in ["val"]:
            object_paths = [p for p in (Path(data_path) / stage).iterdir() if p.is_dir()]
            dataset_list = []
            for object_path in object_paths:
                dataset_list.extend([p for p in object_path.iterdir() if p.is_dir()])
            dataset_.append(
                RelativePoseDataset(
                    config=config,
                    data_list=dataset_list,
                    urdf_path=urdf_path,
                    baseline_signal_path=baseline_signal_path,
                )
            )
        val_dset = dataset_[0]
        return val_dset


def get_xela_downstream_data(cfg: DictConfig):
    data_cfg = cfg.data
    val_dataset = create_from_files(data_cfg.dataset.data_path, data_cfg.dataset.urdf_path, data_cfg.dataset.baseline_signal_path, data_cfg.dataset.config)
    return val_dataset

In [ ]:
with hydra.initialize(version_base="1.3", config_path="../config"):
    config = hydra.compose(
        config_name="experiment/xela/task/relative_pose_estimation/dinov2.yaml",
        overrides=[
            "paths=default",
            "hydra.job.num=1",
            "data.dataset.config.normalize=False",
            "data.train_dataloader.shuffle=False",
            "paths.output_dir='outputs/.'",
            "hydra.runtime.output_dir='outputs/.'",
            "paths.work_dir='outputs/.'",
        ]
    )
print(OmegaConf.to_yaml(config, resolve=True))

val_dset = get_xela_downstream_data(config)


In [28]:
val_dloader = DataLoader(val_dset, batch_size=1, shuffle=False, num_workers=0)

In [ ]:
batch = next(iter(val_dloader))

x = batch['sensor'].to(device)
print(x.shape)
chunked_time = x.shape[1] // encoder.sequence_length
x = einops.rearrange(x, 'b (l k) n c -> (b l) k n c', k=encoder.sequence_length)

z = ssl_encoder.forward_features(x)
print(z['x_norm_regtokens'].shape)
print(z['x_norm_patchtokens'].shape)


In [30]:
def pairwise_similarity(a, b):
    a = a / torch.norm(a, dim=-1, keepdim=True)
    b = b / torch.norm(b, dim=-1, keepdim=True)

    return torch.einsum('ik,jk->ij', a, b)


sim = pairwise_similarity(z['x_norm_regtokens'][:, 0], z['x_norm_regtokens'][:, 0])
sim = sim.cpu().detach().numpy()
sim = sim - np.min(sim) / (np.max(sim) - np.min(sim))
sim = (np.clip(sim, 0, 1) * 255).astype(np.uint8)

In [ ]:
plt.imshow(sim, cmap='gray', vmin=0, vmax=255)
plt.show()
